In [25]:
import torch
import MinkowskiEngine as ME
data = [
    [0, 0, 2.1, 0, 0],
    [0, 1, 1.4, 3, 0],
    [0, 0, 4.0, 0, 0]
]

def to_sparse_coo(data):
    # An intuitive way to extract coordinates and features
    coords, feats = [], []
    for i, row in enumerate(data):
        for j, val in enumerate(row):
            if val != 0:
                coords.append([i, j])
                feats.append([val])
    return torch.IntTensor(coords), torch.FloatTensor(feats)

C, F = to_sparse_coo(data)
sparse_tensor = ME.SparseTensor(features=F, coordinates=C)
sparse_tensor

SparseTensor(
  coordinates=tensor([[0, 2],
        [1, 1],
        [1, 2],
        [1, 3],
        [2, 2]], dtype=torch.int32)
  features=tensor([[2.1000],
        [1.0000],
        [1.4000],
        [3.0000],
        [4.0000]])
  coordinate_map_key=coordinate map key:[1]
  coordinate_manager=CoordinateMapManagerCPU(
	[1, ]:	CoordinateMapCPU:5x2
	algorithm=MinkowskiAlgorithm.DEFAULT
  )
  spatial dimension=1)

In [13]:
import torch
import torch.nn as nn
from torch.optim import SGD

import MinkowskiEngine as ME
import numpy as np

def get_coords(data):
    coords = []
    for i, row in enumerate(data):
        for j, col in enumerate(row):
            if col != " ":
                coords.append([i, j])
    return np.array(coords)
def data_loader(
    nchannel=30,
    max_label=5,
    is_classification=True,
    seed=-1,
    batch_size=2,
    dtype=torch.float32,
):
    if seed >= 0:
        torch.manual_seed(seed)

    data = ["   X   ", "  X X  ", " XXXXX "]

    # Generate coordinates
    coords = [get_coords(data) for i in range(batch_size)]
    coords = ME.utils.batched_coordinates(coords)

    # features and labels
    N = len(coords)
    feats = torch.arange(N * nchannel).view(N, nchannel).to(dtype)
    label = (torch.rand(batch_size if is_classification else N) * max_label).long()
    return coords, feats, label


class ExampleNetwork(ME.MinkowskiNetwork):

    def __init__(self, in_feat, out_feat, D):
        super(ExampleNetwork, self).__init__(D)
        self.net = nn.Sequential(
            ME.MinkowskiConvolution(
                in_channels=in_feat,
                out_channels=64,
                kernel_size=3,
                stride=2,
                dilation=1,
                bias=False,
                dimension=D), ME.MinkowskiBatchNorm(64), ME.MinkowskiReLU(),
            ME.MinkowskiConvolution(
                in_channels=64,
                out_channels=128,
                kernel_size=3,
                stride=2,
                dimension=D), ME.MinkowskiBatchNorm(128), ME.MinkowskiReLU(),
            ME.MinkowskiGlobalPooling(),
            ME.MinkowskiLinear(128, out_feat))

    def forward(self, x):
        return self.net(x)


# loss and network
criterion = nn.CrossEntropyLoss()
net = ExampleNetwork(in_feat=30, out_feat=5, D=2)

# a data loader must return a tuple of coords, features, and labels.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

net = net.to(device)
optimizer = SGD(net.parameters(), lr=1e-1)

for i in range(1):
    optimizer.zero_grad()

    # Get new data
    coords, feat, label = data_loader()
    input = ME.SparseTensor(feat, coords, device=device)
    label = label.to(device)

    # Forward
    output = net(input)
    print(output.F.shape)

    # Loss
    loss = criterion(output.F, label)

    # Gradient
    loss.backward()
    optimizer.step()

RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1.
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [37]:
import argparse
import random
from typing import Any, Dict

import numpy as np
import torch
import torch.utils.data
from torch import nn
from torch.cuda import amp

import torchsparse
from torchsparse import SparseTensor
from torchsparse import nn as spnn
from torchsparse.nn import functional as F
from torchsparse.utils.collate import sparse_collate_fn
from torchsparse.utils.quantize import sparse_quantize
import sys

class RandomDataset:
    def __init__(self, input_size: int, voxel_size: float) -> None:
        self.input_size = input_size
        self.voxel_size = voxel_size

    def __getitem__(self, _: int) -> Dict[str, Any]:
        inputs = np.random.uniform(-100, 100, size=(self.input_size, 4))
        labels = np.random.choice(10, size=self.input_size)

        coords, feats = inputs[:, :3], inputs
        coords -= np.min(coords, axis=0, keepdims=True)
        coords, indices = sparse_quantize(coords, self.voxel_size, return_index=True)

        coords = torch.tensor(coords, dtype=torch.int)
        feats = torch.tensor(feats[indices], dtype=torch.float)
        labels = torch.tensor(labels[indices], dtype=torch.long)

        input = SparseTensor(coords=coords, feats=feats)
        label = SparseTensor(coords=coords, feats=labels)
        return {"input": input, "label": label}

    def __len__(self):
        return 100

random.seed(0)
np.random.seed(0)
torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
amp_enabled=True

dataset = RandomDataset(input_size=10000, voxel_size=0.2)
dataflow = torch.utils.data.DataLoader(
    dataset,
    batch_size=2,
    collate_fn=sparse_collate_fn,
)

model = nn.Sequential(
    spnn.Conv3d(4, 32, 3),
    spnn.BatchNorm(32),
    spnn.ReLU(True),
    spnn.Conv3d(32, 64, 2, stride=2),
    spnn.BatchNorm(64),
    spnn.ReLU(True),
    spnn.Conv3d(64, 64, 2, stride=2, transposed=True),
    spnn.BatchNorm(64),
    spnn.ReLU(True),
    spnn.Conv3d(64, 32, 3),
    spnn.BatchNorm(32),
    spnn.ReLU(True),
    spnn.Conv3d(32, 10, 1)
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scaler = amp.GradScaler(enabled=amp_enabled)

for k, feed_dict in enumerate(dataflow):
    inputs = feed_dict["input"].to(device=device)
    labels = feed_dict["label"].to(device=device)

    with amp.autocast(enabled=amp_enabled):
        outputs = model(inputs)
        print(outputs.F.shape)
        loss = criterion(outputs.feats, labels.feats)

    #print(f"[step {k + 1}] loss = {loss.item()}")

    optimizer.zero_grad()
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()

cuda
torch.Size([20000, 10])
torch.Size([20000, 10])
torch.Size([20000, 10])
torch.Size([20000, 10])
torch.Size([20000, 10])
torch.Size([20000, 10])
torch.Size([20000, 10])
torch.Size([20000, 10])
torch.Size([20000, 10])
torch.Size([20000, 10])
torch.Size([20000, 10])
torch.Size([20000, 10])
torch.Size([20000, 10])
torch.Size([20000, 10])
torch.Size([20000, 10])
torch.Size([20000, 10])
torch.Size([20000, 10])
torch.Size([20000, 10])
torch.Size([20000, 10])
torch.Size([20000, 10])
torch.Size([20000, 10])
torch.Size([19999, 10])
torch.Size([20000, 10])
torch.Size([20000, 10])
torch.Size([20000, 10])
torch.Size([20000, 10])
torch.Size([20000, 10])
torch.Size([20000, 10])
torch.Size([20000, 10])
torch.Size([20000, 10])
torch.Size([20000, 10])
torch.Size([20000, 10])
torch.Size([20000, 10])
torch.Size([20000, 10])
torch.Size([20000, 10])
torch.Size([20000, 10])
torch.Size([20000, 10])
torch.Size([20000, 10])
torch.Size([20000, 10])
torch.Size([20000, 10])
torch.Size([20000, 10])
torch.Size(

In [ ]:
# enable torchsparse 2.0 inference
model.eval()
# enable fused and locality-aware memory access optimization
torchsparse.backends.benchmark = True  # type: ignore

with torch.no_grad():
    for k, feed_dict in enumerate(dataflow):
        inputs = feed_dict["input"].to(device=device).half()
        labels = feed_dict["label"].to(device=device)

        with amp.autocast(enabled=True):
            outputs = model(inputs)
            loss = criterion(outputs.feats, labels.feats)

        print(f"[inference step {k + 1}] loss = {loss.item()}")